# 01 — Catálogo de distribuições (`distributions`) e verificação (`diagnostics`)

Toda distribuição é um dicionário com a **mesma interface**. Os módulos `models`, `separability`, `validation`, `fitting` e `raster_io` acessam as distribuições apenas por essa interface. Por isso, incluir uma distribuição nova não exige alterar nenhum outro módulo.

## Interface

| Campo | Tipo | Uso |
|---|---|---|
| `label` | `str` | nome legível |
| `domain` | `str` | `'intensity'`, `'amplitude'` ou `'any'`; define o suporte nas verificações |
| `n_features` | `int` ou `None` | nº de atributos exigido (`None` = qualquer) |
| `requires_enl` | `bool` | ENL obrigatório na estimação/densidade |
| `fit_methods` | `tuple` | estimadores aceitos; o 1º é o padrão |
| `fit(x, ENL, method)` | → `dict` | parâmetros |
| `logpdf(x, params, ENL)` | → array `(n,)` | log-densidade vetorizada (usada em **toda** a classificação) |
| `cdf(x, params, ENL)` | → array ou `None` | teste KS (univariadas) |
| `sample(n, params, ENL, rng)` | → array `(n, p)` | simulação (testes e dados fictícios) |
| `bhattacharyya(p1, p2, ENL)` | → `float` ou `None` | forma fechada, se existir |

Convenção: `x` é sempre um array 2D (n amostras × p atributos).

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))   # pasta que contém sar_classification

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sar_classification import (distributions as dist, samples as smp, models as mdl,
                                raster_io as rio, separability as sep, validation as val,
                                fitting as fit, plots, diagnostics as diag)

In [2]:
dist.list_distributions()

,name,label,domain,n_features,requires_enl,fit_methods,cdf (KS),bhattacharyya
0,gaussian,Gaussiana multivariada,any,≥ 1,False,"unbiased, mle",True,True
1,gamma,Gama,intensity,1,True,"mle, mme",True,True
2,intensity_joint_distribution,Par de intensidades,intensity,2,True,moments,False,False


## Distribuições cadastradas

**Gaussiana multivariada** (`'gaussian'`), Eq. 2.6:

$$\ln f(\mathbf{x}) = -\tfrac{B}{2}\ln(2\pi) - \tfrac{1}{2}\ln|\Sigma| - \tfrac{1}{2}(\mathbf{x}-\mathbf{m})^\intercal\Sigma^{-1}(\mathbf{x}-\mathbf{m})$$

| `method` | Covariância | Onde é usado |
|---|---|---|
| `'unbiased'` (padrão) | denominador $n-1$, como `cov()` do R | classificação e separabilidade |
| `'mle'` | denominador $n$ | teste KS, como no `fitdistrplus` |

A Bhattacharyya é calculada com $\Sigma = (\Sigma_1+\Sigma_2)/2$:

$$B = \tfrac{1}{8}\Delta^\intercal\Sigma^{-1}\Delta + \tfrac{1}{2}\ln\frac{|\Sigma|}{\sqrt{|\Sigma_1||\Sigma_2|}}$$

**Gama** (`'gamma'`), univariada:

$$f(x) = \frac{\beta^\alpha}{\Gamma(\alpha)}x^{\alpha-1}e^{-\beta x}$$

| Chamada | Estimativa |
|---|---|
| com `ENL` | $\alpha = \text{ENL}$ (fixo), $\beta = \text{ENL}/\bar{x}$ (modelo de classificação) |
| sem `ENL`, `method='mle'` | $\alpha$, $\beta$ livres por máxima verossimilhança |
| sem `ENL`, `method='mme'` | momentos: $\alpha = \bar{x}^2/s^2$, $\beta = \bar{x}/s^2$ ($s^2$ com denominador $n$) |

A Bhattacharyya usa a mesma forma $L$: $\;B = L\ln\frac{\lambda_1+\lambda_2}{2\sqrt{\lambda_1\lambda_2}}$, com $\lambda = \alpha/\beta$.

**Par de intensidades** (`'intensity_joint_distribution'`), com $n$ = ENL (Lee et al., 1994; Correia, 1998):

$$f(I_1,I_2) = \frac{n^{n+1}(I_1I_2)^{\frac{n-1}{2}}\, e^{-\frac{n(I_1/\mu_1+I_2/\mu_2)}{1-|\rho_c|^2}}}{(\mu_1\mu_2)^{\frac{n+1}{2}}\Gamma(n)(1-|\rho_c|^2)|\rho_c|^{n-1}}\, I_{n-1}\!\left(\frac{2n|\rho_c|}{1-|\rho_c|^2}\sqrt{\frac{I_1I_2}{\mu_1\mu_2}}\right)$$

- Os parâmetros são $\mu_1, \mu_2$ (médias) e `ro2` = $|\text{corr}(I_1,I_2)|$, que estima $|\rho_c|^2$.
- A Bessel é calculada em log ($\ln I_\nu(z) = \ln\texttt{ive}(\nu,z)+z$), sem truncamento.
- Com $\rho_c = 0$, vale o limite exato: o produto de duas Gamas.
- Não há cdf (o par é bivariado) nem Bhattacharyya fechada.

## Verificação automática — `diagnostics.check_distribution(name, params, ENL)`

Testa a coerência interna de uma entrada do catálogo, **sem referência externa**:

| Teste | Critério |
|---|---|
| Normalização | $\int f = 1$ (integração numérica, 1D ou 2D), tolerância 10⁻⁴ |
| Simulação × cdf | KS da amostra simulada contra a cdf, p > 0,01 |
| Recuperação | `fit(sample(params)) ≈ params`, erro relativo < 5% |
| Bhattacharyya | forma fechada × $-\ln\int\sqrt{f_1f_2}$ numérica |

Para o par de intensidades, a normalização confere a fórmula da densidade de forma independente, e a recuperação de `ro2` confirma que a correlação das intensidades estima $|\rho_c|^2$.

In [3]:
casos = [('gaussian', {'mean': np.array([0.2]), 'covariance_matrix': np.array([[9e-4]])}, None),
         ('gamma', {'alpha': 3.0, 'beta': 60.0}, 3.0),
         ('intensity_joint_distribution', {'mu1': 0.08, 'mu2': 0.02, 'ro2': 0.25}, 4)]
for name, params, ENL in casos:
    print(f'--- {name}')
    print(diag.check_distribution(name, params, ENL).to_string(index=False), '\n')

--- gaussian
                           teste resultado                               detalhe
             Normalização ∫f = 1        OK                              1.000000
            Simulação × cdf (KS)        OK                 D = 0.0071, p = 0.264
      Recuperação dos parâmetros        OK mean: 0.07%, covariance_matrix: 0.79%
Bhattacharyya fechada × numérica        OK                   0.439073 × 0.439073 

--- gamma
                           teste resultado                   detalhe
             Normalização ∫f = 1        OK                  1.000000
            Simulação × cdf (KS)        OK     D = 0.0063, p = 0.410
      Recuperação dos parâmetros        OK alpha: 0.00%, beta: 0.34%
Bhattacharyya fechada × numérica        OK       0.025739 × 0.025739 

--- intensity_joint_distribution


                     teste resultado                            detalhe
       Normalização ∫f = 1        OK                           1.000000
Recuperação dos parâmetros        OK mu1: 0.37%, mu2: 0.48%, ro2: 1.40% 



## Como cadastrar uma distribuição nova

1. Escreva as funções com as assinaturas da interface (`fit`, `logpdf` e, se houver, `cdf`, `sample`, `bhattacharyya`), calculando a densidade **em log**.
2. Registre a entrada em `DISTRIBUTIONS` em `distributions.py`.
3. Rode `diagnostics.check_distribution` com parâmetros conhecidos.

A partir daí, a distribuição já funciona em `fit_model`, `class_likelihood`, `cross_validation`, `likelihood_raster`, `fit_and_test` (se tiver `cdf`) e `bhattacharyya_matrix` (se tiver forma fechada).

```python
def _nova_fit(x, ENL=None, method='mle'):        ...  # -> dict de parâmetros
def _nova_logpdf(x, params, ENL=None):           ...  # -> array (n,)
def _nova_cdf(x, params, ENL=None):              ...  # -> array (n,) ou omitir (None)
def _nova_sample(n, params, ENL=None, rng=None): ...  # -> array (n, p)

DISTRIBUTIONS['nova'] = {
    'label': 'Nome legível', 'domain': 'intensity', 'n_features': 1,
    'requires_enl': True, 'fit_methods': ('mle',),
    'fit': _nova_fit, 'logpdf': _nova_logpdf, 'cdf': _nova_cdf,
    'sample': _nova_sample, 'bhattacharyya': None,
}
```